# t-SNE evaluation for multiple time-series distances (UCR/UEA via tslearn)
Self-contained notebook: distance methods + distance-matrix builder + t-SNE quality metrics + plotting + CSV export.


In [ ]:
# Kaggle: run once (restart runtime if Kaggle asks)
# Sá»­ dá»¥ng cÃ¡c phiÃªn báº£n tÆ°Æ¡ng thÃ­ch vá»›i Python 3.12
!pip -q install --upgrade pip
!pip -q uninstall -y numpy scipy pandas matplotlib scikit-learn scikit-learn-extra tslearn POT
# CÃ i NumPy 2.x vÃ  cÃ¡c packages tÆ°Æ¡ng thÃ­ch
!pip -q install numpy>=2.0.0
!pip -q install scipy>=1.13.0 pandas>=2.2.0 matplotlib>=3.8.0
!pip -q install scikit-learn>=1.4.0 scikit-learn-extra>=0.3.0
!pip -q install tslearn>=0.6.3
# POT 0.9.5+ há»— trá»£ NumPy 2.x
!pip -q install POT>=0.9.5

   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1.8/1.8 MB 30.9 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.4.0 requires scipy>=1.7.0, which is not installed.
datasets 4.4.1 requires pandas, which is not installed.
woodwork 0.31.0 requires pandas>=2.0.0, which is not installed.
woodwork 0.31.0 requires scikit-learn>=1.1.0, which is not installed.
woodwork 0.31.0 requires scipy>=1.10.0, which is not installed.
cartopy 0.25.0 requires matplotlib>=3.6, which is not installed.
boruta 0.4.3 requires scikit-learn>=0.17.1, which is not installed.
boruta 0.4.3 requires scipy>=0.17.0, which is not installed.
kauldron 1.3.0 requires pandas, which is not installed.
kauldron 1.3.0 requires scikit-learn, which is not installed.
scikit-surprise 1.1.4 requires scipy>

## 1) Distance methods (copied from kmedoids.ipynb)

In [ ]:
# otsw_api.py  — RAGGED-FRIENDLY
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Sequence, Union
import heapq

# (Optional) Use SciPy to accelerate SpMM; works without it as well
try:
    import scipy.sparse as sp
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

BIG = 1e12

# =========================
# 0) HELPERS (ragged / dense)
# =========================
def _as_ragged_list(M: Union[np.ndarray, Sequence[np.ndarray]]) -> Tuple[List[np.ndarray], int]:
    """
    Normalize input to a list of arrays (n_i, d).
    Returns (list_seq, d)
    """
    if isinstance(M, np.ndarray):
        if M.ndim != 3:
            raise ValueError("If ndarray, expect shape (m, n, d).")
        m, n, d = M.shape
        seqs = [M[i] for i in range(m)]
        return seqs, d
    # list/tuple các chuỗi (n_i, d)
    seqs = []
    d = None
    for i, xi in enumerate(M):
        xi = np.asarray(xi, dtype=float)
        if xi.ndim != 2:
            raise ValueError(f"Sequence {i} must have shape (n_i, d).")
        if d is None:
            d = xi.shape[1]
        elif xi.shape[1] != d:
            raise ValueError("All sequences must have the same feature dimension d.")
        seqs.append(xi)
    if d is None:
        raise ValueError("Empty sequence list.")
    return seqs, d

def _linearize_points_ragged(M: Union[np.ndarray, Sequence[np.ndarray]]):
    """
    Ragged support: 
      - P: (N, d) concatenated points
      - Sidx: (N,) series id
      - Tpos: (N,) normalized time in [0,1) for each point (i / n_i)
      - lengths: (m_seq,) length of each series
      - d: number of channels
    """
    seqs, d = _as_ragged_list(M)
    m_seq = len(seqs)
    lengths = np.array([xi.shape[0] for xi in seqs], dtype=int)
    # concatenate points
    P = np.vstack(seqs) if m_seq > 0 else np.zeros((0, d))
    # series id
    Sidx = np.repeat(np.arange(m_seq, dtype=int), lengths)
    # normalized time position (avoid endpoint=1 to prevent collision at 1.0)
    Tpos_list = [ (np.arange(n_i, dtype=float) / max(n_i,1)) for n_i in lengths ]
    Tpos = np.concatenate(Tpos_list) if m_seq > 0 else np.zeros((0,), float)
    return P, Sidx, Tpos, m_seq, lengths, d

def _pairwise_sqdist(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    'Hybrid' distance:
      Euclid^2 on features (except last column)  +  |orderA - orderB| (last column)
    """
    assert A.ndim == 2 and B.ndim == 2, "Expect 2D arrays"
    assert A.shape[1] == B.shape[1], "Dim mismatch"
    D = A.shape[1]
    if D == 0:
        return np.zeros((A.shape[0], B.shape[0]), dtype=float)
    if D == 1:
        a_ord = A[:, 0]; b_ord = B[:, 0]
        return np.abs(a_ord[:, None] - b_ord[None, :])
    Af = A[:, :-1]; Bf = B[:, :-1]
    aa = (Af * Af).sum(1)[:, None]
    bb = (Bf * Bf).sum(1)[None, :]
    Dsq = np.clip(aa + bb - 2 * (Af @ Bf.T), 0.0, None)
    a_ord = A[:, -1]; b_ord = B[:, -1]
    Pen = np.abs(a_ord[:, None] - b_ord[None, :])
    return Dsq + Pen

@dataclass
class _Node:
    idx: np.ndarray
    height: float
    left: Optional[int]
    right: Optional[int]
    parent: Optional[int]
    is_leaf: bool

@dataclass
class OTSWModel:
    # shared runtime fields
    P: np.ndarray                  # (N, d_aug) nếu TamLe, hoặc (N, d) nếu Banded
    Sidx: np.ndarray               # (N,) id chuỗi
    Tpos: np.ndarray               # (N,) thời gian chuẩn hoá (0..1)
    lengths: np.ndarray            # (m_seq,)
    m_seq: int
    d: int                         # số kênh gốc (chưa augment)
    nodes: List[_Node]
    leaf_ids: List[int]
    leaf_index_map: Dict[int, int]
    edges: List[Tuple[int, int, float]]     # (parent, child, w_e)
    S_edge_leaf: object                     # (E, L) dense hoặc sp.csr_matrix
    centroids: np.ndarray                   # (num_nodes, D_aug)
    # meta
    mode: str                               # "tamle" | "banded"
    lam_time: float = 0.0
    lam_idx: float = 0.0
    W: float = 0.0                           # band width (tỉ lệ 0..1 cho ragged)
    # caches
    point_leaf: Optional[np.ndarray] = None  # (N,)
    H: Optional[np.ndarray] = None           # (L, m_seq)
    M: Optional[np.ndarray] = None           # (E, m_seq)
    w: Optional[np.ndarray] = None           # (E,)

# =========================
# 1) BOX TREE + GONZALEZ
# =========================
class _KDBoxTree:
    def __init__(self, leaf_size=64, max_depth=24):
        self.leaf_size = leaf_size
        self.max_depth = max_depth
        self.boxes = []
        self.X = None

    def _build(self, idx, depth):
        X = self.X[idx]
        c = X.mean(axis=0)
        r = float(np.sqrt(((X - c) ** 2).sum(1).max())) if X.shape[0] else 0.0
        bid = len(self.boxes)
        self.boxes.append({"idx": idx, "c": c, "r": r, "L": None, "R": None, "leaf": False})
        if idx.size <= self.leaf_size or depth >= self.max_depth or r == 0.0:
            self.boxes[bid]["leaf"] = True
            return bid
        var = X.var(axis=0)
        d = int(np.argmax(var))
        med = np.median(X[:, d])
        mask = X[:, d] <= med
        if mask.all() or (~mask).all():
            mid = idx.size // 2
            Lidx = idx[:mid]; Ridx = idx[mid:]
        else:
            Lidx = idx[mask]; Ridx = idx[~mask]
        L = self._build(Lidx, depth + 1)
        R = self._build(Ridx, depth + 1)
        self.boxes[bid]["L"] = L; self.boxes[bid]["R"] = R
        return bid

    def fit(self, X):
        self.X = X
        self.boxes = []
        self._build(np.arange(X.shape[0]), 0)

def _bounds_box(box, centers):
    if centers.size == 0: return 0.0, float("inf")
    d = np.sqrt(((centers - box["c"][None, :]) ** 2).sum(1))
    dmin = float(d.min())
    r = box["r"]
    return max(0.0, dmin - r), dmin + r

def _farthest_point_by_boxes(X, centers, kdt: _KDBoxTree, gap_tol=1e-6):
    if centers.size == 0: return 0, 0.0
    heap = []
    L0, U0 = _bounds_box(kdt.boxes[0], centers)
    heapq.heappush(heap, (-U0, 0, L0))
    best_idx, best_val = None, -1.0
    while heap:
        negU, bid, Lb = heapq.heappop(heap)
        Ub = -negU
        L2, U2 = _bounds_box(kdt.boxes[bid], centers)
        if U2 < Ub - 1e-12 or L2 > Lb + 1e-12:
            heapq.heappush(heap, (-U2, bid, L2)); continue
        if best_val >= U2 - 1e-15: break
        box = kdt.boxes[bid]
        if box["leaf"] or (U2 - L2) <= gap_tol:
            pts = kdt.X[box["idx"]]
            D = _pairwise_sqdist(pts, centers)  # KHÔNG sqrt
            dmin = D.min(axis=1)
            imax = int(np.argmax(dmin)); val = float(dmin[imax])
            if val > best_val: best_val, best_idx = val, int(box["idx"][imax])
            continue
        for child in (box["L"], box["R"]):
            Lc, Uc = _bounds_box(kdt.boxes[child], centers)
            heapq.heappush(heap, (-Uc, child, Lc))
    return best_idx, best_val

def _gonzalez_box_nlogk(X: np.ndarray, k: int, seed: int,
                        box_leaf_size=64, box_max_depth=24, gap_tol=1e-6):
    rng = np.random.default_rng(seed)
    n = X.shape[0]; assert 1 <= k <= n
    kdt = _KDBoxTree(leaf_size=box_leaf_size, max_depth=box_max_depth); kdt.fit(X)
    i0 = int(rng.integers(0, n)); centers = X[i0:i0+1]; C = [i0]
    for _ in range(1, k):
        idx, _ = _farthest_point_by_boxes(X, centers, kdt, gap_tol)
        C.append(idx); centers = X[np.array(C)]
    return np.array(C, dtype=int)

# =========================
# 1.1) ROUTING & PRECOMPUTE
# =========================
def _route_all_points_vectorized(model: OTSWModel) -> np.ndarray:
    N = model.P.shape[0]
    leaf_of_point = np.empty(N, dtype=np.int32)
    stack = [(0, np.arange(N, dtype=np.int32))]
    nodes = model.nodes; C = model.centroids; P = model.P
    while stack:
        nid, idxs = stack.pop()
        nd = nodes[nid]
        if nd.is_leaf:
            j = model.leaf_index_map[nid]; leaf_of_point[idxs] = j; continue
        L = nd.left; R = nd.right
        X = P[idxs]
        child_centroids = np.vstack([C[L], C[R]])
        d_lr = _pairwise_sqdist(X, child_centroids)
        go_left = d_lr[:, 0] <= d_lr[:, 1]
        if go_left.any():    stack.append((L, idxs[go_left]))
        if (~go_left).any(): stack.append((R, idxs[~go_left]))
    return leaf_of_point

def _precompute_H_M(model: OTSWModel):
    m_seq = model.m_seq
    N = model.P.shape[0]
    L = len(model.leaf_ids)
    E = len(model.edges)
    # 1) route tất cả điểm -> lá
    point_leaf = _route_all_points_vectorized(model)  # (N,)
    model.point_leaf = point_leaf
    # 2) H (L, m_seq) — histogram mỗi chuỗi
    H = np.zeros((L, m_seq), dtype=np.float32)
    for s in range(m_seq):
        mask = (model.Sidx == s)
        if not np.any(mask): continue
        counts = np.bincount(point_leaf[mask], minlength=L).astype(np.float32)
        tot = counts.sum()
        if tot > 0: counts /= tot
        H[:, s] = counts
    model.H = H
    # 3) S_edge_leaf -> CSR (nếu có SciPy) và M = S @ H
    if _HAS_SCIPY:
        SpS = sp.csr_matrix(model.S_edge_leaf)
        model.S_edge_leaf = SpS
        M = (SpS @ H).astype(np.float32)  # (E, m_seq)
    else:
        M = (model.S_edge_leaf @ H).astype(np.float32)
    model.M = M
    # 4) Trọng số cạnh
    model.w = np.array([we for _, _, we in model.edges], dtype=np.float32)

# =========================
# 1.2) OTSW — TAM LE (ragged OK)
# =========================
def _augment_points(seq: np.ndarray, lam_time: float) -> np.ndarray:
    n = seq.shape[0]
    t = (np.arange(n, dtype=float) / max(n, 1))[:, None] * np.sqrt(lam_time)
    return np.hstack([seq, t])

def build_otsw_tamle(
    M: Union[np.ndarray, Sequence[np.ndarray]],
    lam_time: float = 5.0,
    leaf_size: int = 16,
    max_depth: int = 20,
    seed: int = 0,
    k_split: int = 2,
    box_leaf_size: int = 64,
    box_max_depth: int = 24,
) -> OTSWModel:
    """
    Xây cây global theo TamLe (augment theo thời gian chuẩn hoá → ragged friendly).
    """
    P_raw, Sidx, Tpos, m_seq, lengths, d = _linearize_points_ragged(M)
    # augment từng chuỗi rồi ghép
    P_aug_list = []
    start = 0
    for s in range(m_seq):
        n_i = lengths[s]
        seq = P_raw[start:start+n_i]
        P_aug_list.append(_augment_points(seq, lam_time))
        start += n_i
    P_aug = np.vstack(P_aug_list) if P_aug_list else np.zeros((0, d+1))
    # build tree
    nodes: List[_Node] = []; leaf_ids: List[int] = []
    def _hybrid_radius(X):
        if X.shape[0] <= 1: return 0.0
        if X.shape[0] > 1024:
            I = np.random.default_rng(0).choice(X.shape[0], 1024, replace=False); Y = X[I]
        else: Y = X
        j0 = 0; d0 = _pairwise_sqdist(Y, Y[j0:j0+1]).reshape(-1); j1 = int(np.argmax(d0))
        d1 = _pairwise_sqdist(Y, Y[j1:j1+1]).reshape(-1); return 0.5 * float(d1.max())
    def build(idx: np.ndarray, depth: int, parent: Optional[int], seed_: int) -> int:
        Xsub = P_aug[idx]; h = _hybrid_radius(Xsub)
        nid = len(nodes); nodes.append(_Node(idx, h, None, None, parent, False))
        if idx.size <= leaf_size or depth >= max_depth or h == 0.0:
            nodes[nid].is_leaf = True; leaf_ids.append(nid); return nid
        C = _gonzalez_box_nlogk(Xsub, k=k_split, seed=seed_,
                                box_leaf_size=box_leaf_size, box_max_depth=box_max_depth)
        centers = Xsub[C]
        lab = np.argmin(_pairwise_sqdist(Xsub, centers), axis=1)
        if k_split == 2:
            left_idx = idx[lab == 0]; right_idx = idx[lab != 0]
        else:
            cnt = np.bincount(lab, minlength=k_split); main = int(np.argmax(cnt))
            left_idx = idx[lab == main]; right_idx = idx[lab != main]
        if left_idx.size == 0 or right_idx.size == 0:
            mid = idx.size // 2; left_idx = idx[:mid]; right_idx = idx[mid:]
        L = build(left_idx, depth+1, nid, seed_+1); R = build(right_idx, depth+1, nid, seed_+2)
        nodes[nid].left, nodes[nid].right = L, R; return nid
    _ = build(np.arange(P_aug.shape[0]), 0, None, seed)
    # edges & structures
    edges = []
    for cid, nd in enumerate(nodes):
        if nd.parent is not None:
            p = nodes[nd.parent]; w = max(0.0, p.height - nd.height)
            edges.append((nd.parent, cid, w))
    leaf_index_map = {nid: i for i, nid in enumerate(leaf_ids)}
    E, L = len(edges), len(leaf_ids)
    S_edge_leaf = np.zeros((E, L), dtype=np.float32)
    def collect_leaves(nid, out):
        nd = nodes[nid]
        if nd.is_leaf: out.append(nid); return
        if nd.left is not None: collect_leaves(nd.left, out)
        if nd.right is not None: collect_leaves(nd.right, out)
    for e, (pid, cid, _) in enumerate(edges):
        leaves = []; collect_leaves(cid, leaves)
        for ln in leaves:
            j = leaf_index_map[ln]; S_edge_leaf[e, j] = 1.0
    centroids = np.vstack([P_aug[nd.idx].mean(axis=0) for nd in nodes])
    model = OTSWModel(
        P=P_aug, Sidx=Sidx, Tpos=Tpos, lengths=lengths, m_seq=m_seq, d=d,
        nodes=nodes, leaf_ids=leaf_ids, leaf_index_map=leaf_index_map,
        edges=edges, S_edge_leaf=S_edge_leaf, centroids=centroids,
        mode="tamle", lam_time=lam_time
    )
    _precompute_H_M(model)
    return model

# =========================
# 3) DISTANCE APIs
# =========================
def otsw_between_series_fast(model: OTSWModel, s_ref: int, s_cmp: int) -> float:
    """
    OTSW(s_ref, s_cmp) với cache:
      cost = sum_e w_e * |M[e, s_ref] - M[e, s_cmp]|
    """
    w = model.w; M = model.M
    diff = np.abs(M[:, s_ref] - M[:, s_cmp])
    return float((w * diff).sum())

def otsw_between_series(model: OTSWModel, s_ref: int, s_cmp: int, p: int = 1) -> float:
    assert p == 1, "Hiện tại hỗ trợ p=1 (W1    cây)."
    return otsw_between_series_fast(model, s_ref, s_cmp)

In [ ]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def asw_distance(
    A,
    B,
    lam=10.0,          # nhÆ° TAOT: reg = 1 / lam cho Sinkhorn
    auto_weight=True,  # báº­t/táº¯t auto-weight cho 3 term
    w_spatial=1.0,     # weight cho spatial khi khÃ´ng auto
    w_order=1.0,       # weight cho order khi khÃ´ng auto
    w_struct=1.0,      # weight cho structural khi khÃ´ng auto
    tolerance=5e-3,
):
    """
    ASW distance (phiÃªn báº£n gáº§n Ä‘Ãºng theo tinh tháº§n Auto-weighted Sequential Wasserstein),
    há»— trá»£ GPU náº¿u A hoáº·c B lÃ  cupy.ndarray.

    Ã tÆ°á»Ÿng:
    - C_spatial(i,j) = ||x_i - y_j||^2
    - C_order(i,j)   = (t_i - s_j)^2  vá»›i t_i, s_j lÃ  vá»‹ trÃ­ chuáº©n hoÃ¡ trong [0,1]
    - C_struct(i,j)  = ||gA_i - gB_j||^2, trong Ä‘Ã³ gA_i, gB_j lÃ  "gradient/cáº¥u trÃºc lÃ¢n cáº­n"
      (chÃªnh lá»‡ch giá»¯a pháº§n tá»­ hiá»‡n táº¡i vÃ  pháº§n tá»­ trÆ°á»›c nÃ³).

    - Tá»•ng cost: C = w_s * C_spatial + w_o * C_order + w_n * C_struct
      vá»›i bá»™ weight cÃ³ thá»ƒ:
        + auto_weight=True: w_s, w_o, w_n láº¥y tá»± Ä‘á»™ng tá»« dá»¯ liá»‡u (xáº¥p xá»‰ ASW gá»‘c),
        + auto_weight=False: dÃ¹ng w_spatial, w_order, w_struct do ngÆ°á»i dÃ¹ng cung cáº¥p.

    Sau Ä‘Ã³ giáº£i Sinkhorn nhÆ° TAOT/POW:
      reg = 1 / lam
      asw = sum_{i,j} P_ij * C_ij
    """

    # 0) Backend: NumPy (CPU) hoáº·c CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # 1) ÄÆ°a dá»¯ liá»‡u vá» Ä‘Ãºng backend, Ä‘áº£m báº£o shape (n,d)
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # ===== 2) C_spatial: khoáº£ng cÃ¡ch Ä‘áº·c trÆ°ng =====
    diff = A[:, None, :] - B[None, :, :]
    C_spatial = xp.sum(diff * diff, axis=2)  # (n, m)

    # ===== 3) C_order: khoáº£ng cÃ¡ch vá»‹ trÃ­ (index) =====
    # Chuáº©n hoÃ¡ index vá» [0,1]
    if n > 1:
        t = xp.linspace(0.0, 1.0, n)
    else:
        t = xp.zeros(1, dtype=xp.float64)

    if m > 1:
        s = xp.linspace(0.0, 1.0, m)
    else:
        s = xp.zeros(1, dtype=xp.float64)

    C_order = (t[:, None] - s[None, :]) ** 2  # (n, m)

    # ===== 4) C_struct: khoáº£ng cÃ¡ch cáº¥u trÃºc/gradient =====
    # Gradient Ä‘Æ¡n giáº£n: gA[i] = A[i] - A[i-1], vá»›i gA[0] = 0
    gA = xp.zeros_like(A)
    if n > 1:
        gA[1:] = A[1:] - A[:-1]

    gB = xp.zeros_like(B)
    if m > 1:
        gB[1:] = B[1:] - B[:-1]

    gdiff = gA[:, None, :] - gB[None, :, :]
    C_struct = xp.sum(gdiff * gdiff, axis=2)  # (n, m)

    # ===== 5) Auto-weight hay dÃ¹ng weight tay =====
    def _mean_safe(M):
        mu = xp.mean(M)
        if not xp.isfinite(mu) or float(mu) == 0.0:
            return 1.0
        return float(mu)

    if auto_weight:
        # Trá»ng sá»‘ tá»‰ lá»‡ nghá»‹ch vá»›i Ä‘á»™ lá»›n trung bÃ¬nh cá»§a tá»«ng term:
        # term nÃ o lá»›n quÃ¡ -> weight nhá» láº¡i, Ä‘á»ƒ cÃ¡c term cÃ¢n báº±ng hÆ¡n.
        mu_s = _mean_safe(C_spatial)
        mu_o = _mean_safe(C_order)
        mu_n = _mean_safe(C_struct)

        ws = 1.0 / mu_s
        wo = 1.0 / mu_o
        wn = 1.0 / mu_n
    else:
        ws = float(w_spatial)
        wo = float(w_order)
        wn = float(w_struct)

    C_base = ws * C_spatial + wo * C_order + wn * C_struct  # (n, m)

    # ===== 6) Chuáº©n hoÃ¡ cost cho Sinkhorn (giá»‘ng TAOT/POW) =====
    med = xp.median(C_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0

    C = C_base / med

    # ===== 7) Khá»‘i lÆ°á»£ng Ä‘á»u + Sinkhorn =====
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # ===== 8) ASW distance =====
    distance_raw = float(xp.sum(P * C_base))
    return distance_raw


/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():
2025-12-26 14:00:29.388553: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766757629.799756      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766757629.916085      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766757630.935519      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766757630.935567      55 computation_placer.cc:177] computation placer already registered.

In [ ]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def taot_distance(A, B, lam=10.0, w=10.0, tolerance=5e-3):
    """
    TAOT distance, há»— trá»£ GPU náº¿u A hoáº·c B lÃ  cupy.ndarray.

    - Náº¿u A/B lÃ  numpy array hoáº·c list -> cháº¡y trÃªn CPU (NumPy).
    - Náº¿u A hoáº·c B lÃ  cupy.ndarray -> convert cáº£ hai sang CuPy vÃ  cháº¡y Sinkhorn trÃªn GPU.
    API (tham sá»‘/hÃ m tráº£ vá») giá»¯ nguyÃªn.
    """
    # Chá»n backend: NumPy (CPU) hoáº·c CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # ÄÆ°a dá»¯ liá»‡u vá» Ä‘Ãºng backend
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # Chuáº©n hoÃ¡ chá»‰ sá»‘ thá»i gian t,s (z-score) báº±ng backend xp
    t = xp.linspace(1, n, n)
    s = xp.linspace(1, m, m)

    def _zscore(x):
        mu = x.mean()
        std = x.std()
        if float(std) == 0.0:
            return x * 0.0
        return (x - mu) / std

    t = _zscore(t)
    s = _zscore(s)

    # Ma tráº­n cost M (data + term báº£o toÃ n thá»© tá»±)
    diff = A[:, None, :] - B[None, :, :]
    M = xp.sum(diff * diff, axis=2) + w * (t[:, None] - s[None, :]) ** 2

    # Chuáº©n hoÃ¡ Ä‘á»ƒ Ä‘Æ°a vá» C cho Sinkhorn
    med = xp.median(M)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    C = M / med

    # Khá»‘i lÆ°á»£ng Ä‘á»u
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    # POT sáº½ tá»± nháº­n backend dá»±a trÃªn kiá»ƒu máº£ng (NumPy hoáº·c CuPy)
    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # TÃ­nh distance trong cÃ¹ng backend rá»“i Ã©p vá» float Python
    distance_raw = float(xp.sum(P * M))
    return distance_raw


In [ ]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def pow_distance(
    A,
    B,
    lam=10.0,          # giá»‘ng TAOT: Ä‘iá»u khiá»ƒn reg = 1/lam cho Sinkhorn
    lam_order=1.0,     # há»‡ sá»‘ cho regularization theo thá»© tá»± (lambda_1)
    bandwidth=None,    # bÄƒng |i-j| cho phÃ©p match; náº¿u None sáº½ auto
    tolerance=5e-3,
):
    """
    POW distance (phiÃªn báº£n gáº§n Ä‘Ãºng theo tinh tháº§n Partial Ordered Wasserstein),
    há»— trá»£ GPU náº¿u A hoáº·c B lÃ  cupy.ndarray.

    Ã tÆ°á»Ÿng:
    - Ground cost: ||x_i - y_j||^2 + lam_order * |i/n - j/m|
    - Chá»‰ cho phÃ©p match trong má»™t bÄƒng theo chá»‰ sá»‘ |i - j| <= bandwidth
      (mÃ´ phá»ng "partial" + háº¡n cháº¿ match xa).
    - DÃ¹ng Sinkhorn (entropic OT) nhÆ° TAOT, reg = 1 / lam.

    Tham sá»‘:
      A, B      : (n,d), (m,d) NumPy hoáº·c CuPy array.
      lam       : tham sá»‘ entropic (reg = 1/lam).
      lam_order : trá»ng sá»‘ cho thÃ nh pháº§n báº£o toÃ n thá»© tá»±.
      bandwidth : náº¿u None -> auto = 0.25 * max(n, m); náº¿u lÃ  sá»‘ nguyÃªn -> dÃ¹ng trá»±c tiáº¿p.
      tolerance : ngÆ°á»¡ng dá»«ng Sinkhorn.

    Tráº£ vá»:
      distance_raw : scalar float (Python float).
    """
    # Chá»n backend: NumPy (CPU) hoáº·c CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # ÄÆ°a dá»¯ liá»‡u vá» Ä‘Ãºng backend
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    # Äáº£m báº£o dáº¡ng (n, d)
    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # 1) Cost Ä‘áº·c trÆ°ng: ||x_i - y_j||^2
    diff = A[:, None, :] - B[None, :, :]
    M_base = xp.sum(diff * diff, axis=2)  # shape (n, m)

    # 2) Regularization tuyáº¿n tÃ­nh theo thá»© tá»±: lam_order * |i/n - j/m|
    #    (khÃ¡c vá»›i TAOT: dÃ¹ng |.|, khÃ´ng bÃ¬nh phÆ°Æ¡ng)
    posA = xp.linspace(1, n, n) / float(n)   # 1/n, 2/n, ..., 1
    posB = xp.linspace(1, m, m) / float(m)   # 1/m, ..., 1
    order_term = xp.abs(posA[:, None] - posB[None, :])

    M_base = M_base + lam_order * order_term

    # 3) Bandwidth (partial constraint): chá»‰ cho phÃ©p match trong |i - j| <= bandwidth
    #    Náº¿u khÃ´ng cho, ta Ä‘áº·t chi phÃ­ Sinkhorn ráº¥t lá»›n á»Ÿ ngoÃ i bÄƒng.
    if bandwidth is None:
        # auto: 1/4 Ä‘á»™ dÃ i lá»›n hÆ¡n, lÃ m int >= 1
        max_len = max(n, m)
        bandwidth = max(1, int(0.25 * max_len))

    idx_i = xp.arange(n)[:, None]
    idx_j = xp.arange(m)[None, :]
    mask = (xp.abs(idx_i - idx_j) <= bandwidth)  # True náº¿u Ä‘Æ°á»£c phÃ©p match

    # 4) Chuáº©n hoÃ¡ cost cho Sinkhorn
    med = xp.median(M_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0

    C = M_base / med

    # ThÃªm penalty ráº¥t lá»›n á»Ÿ ngoÃ i bÄƒng (chá»‰ trong C dÃ¹ng cho Sinkhorn)
    # Ä‘á»ƒ gáº§n nhÆ° cáº¥m mass Ä‘i ra ngoÃ i vÃ¹ng cho phÃ©p.
    big_C = 1e3
    C = C + (~mask).astype(xp.float64) * big_C

    # 5) Khá»‘i lÆ°á»£ng Ä‘á»u
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    # 6) Sinkhorn (POT tá»± nháº­n backend tá»« kiá»ƒu máº£ng)
    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # 7) TÃ­nh distance dÃ¹ng cost "tháº­t" M_base (khÃ´ng cá»™ng big_C)
    distance_raw = float(xp.sum(P * M_base))
    return distance_raw


In [ ]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def tcot_distance_series(x, y, lambda_pos: float = 1.0, reg: float = 0.1, num_iter: int = 1000):
    """
    TÃ­nh khoáº£ng cÃ¡ch Temporally Coupled Optimal Transport (TCOT) giá»¯a hai chuá»—i x, y.

    Há»— trá»£:
      - CPU: náº¿u x, y lÃ  numpy.ndarray (hoáº·c list)  -> dÃ¹ng NumPy + ot.sinkhorn
      - GPU: náº¿u x hoáº·c y lÃ  cupy.ndarray          -> dÃ¹ng CuPy + ot.gpu.sinkhorn

    Parameters
    ----------
    x, y : array-like hoáº·c np.ndarray / cp.ndarray
        - 1D: shape (n,)
        - 2D: shape (n, d)

    lambda_pos : float
        Há»‡ sá»‘ pháº¡t lá»‡ch thá»i gian.

    reg : float
        Entropic regularization cho Sinkhorn.

    num_iter : int
        Sá»‘ vÃ²ng láº·p tá»‘i Ä‘a cho Sinkhorn.

    Returns
    -------
    float
        GiÃ¡ trá»‹ OT(C) vá»›i ground cost TCOT.
    """
    # Chá»n backend: NumPy (CPU) hoáº·c CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(x, (cp.ndarray,)) or isinstance(y, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # ÄÆ°a dá»¯ liá»‡u vá» Ä‘Ãºng backend
    x = xp.asarray(x, dtype=xp.float64)
    y = xp.asarray(y, dtype=xp.float64)

    # Ã‰p vá» dáº¡ng (n, d)
    if x.ndim == 1:
        x = x[:, None]
    if y.ndim == 1:
        y = y[:, None]

    n, dx = x.shape
    m, dy = y.shape
    if dx != dy:
        raise ValueError(f"Dimension mismatch: x dim={dx}, y dim={dy}")
    if n == 0 or m == 0:
        raise ValueError("Empty time series")

    # Cost Ä‘áº·c trÆ°ng: ||x_i - y_j||^2
    diff = x[:, None, :] - y[None, :, :]   # (n, m, d)
    C_feat = xp.sum(diff * diff, axis=2)   # (n, m)

    # Vá»‹ trÃ­ thá»i gian t, s chuáº©n hÃ³a [0,1]
    if n > 1:
        t = xp.linspace(0.0, 1.0, n)
    else:
        t = xp.zeros(1, dtype=xp.float64)

    if m > 1:
        s = xp.linspace(0.0, 1.0, m)
    else:
        s = xp.zeros(1, dtype=xp.float64)

    pos_diff = xp.abs(t[:, None] - s[None, :])  # (n, m)
    C_time = lambda_pos * (pos_diff ** 2)

    # Ground cost TCOT
    C_base = C_feat + C_time
    
    # Chuáº©n hÃ³a cost cho Sinkhorn (giá»‘ng TAOT)
    med = xp.median(C_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    C = C_base / med

    # Trá»ng sá»‘ Ä‘á»u
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    # ThÃªm stopThr Ä‘á»ƒ tÄƒng tá»‘c (giá»‘ng TAOT)
    G = ot.sinkhorn(a, b, C, reg, numItermax=num_iter, stopThr=5e-3)
    cost = float(xp.sum(G * C_base))  # DÃ¹ng C_base (chÆ°a chuáº©n hÃ³a) Ä‘á»ƒ tÃ­nh distance

    return cost


In [ ]:
import numpy as np
import ot

def _get_xp(use_gpu="auto"):
    if use_gpu is False:
        return np, False
    try:
        import cupy as cp
        # ensure CUDA is usable (otherwise fall back to numpy)
        try:
            _ = cp.cuda.runtime.getDeviceCount()
            return cp, True
        except Exception:
            if use_gpu is True:
                raise
            return np, False
    except Exception:
        if use_gpu is True:
            raise
        return np, False


def opw_distance_series(
    x, y,
    lambda1=1.0,
    lambda2=0.1,
    sigma=0.1,
    use_gpu="auto",
    max_iter=1000,
    tol=5e-3,
):
    xp, on_gpu = _get_xp(use_gpu)

    x = xp.asarray(x, dtype=xp.float64)
    y = xp.asarray(y, dtype=xp.float64)
    if x.ndim == 1: x = x[:, None]
    if y.ndim == 1: y = y[:, None]

    Nx, _ = x.shape
    Ny, _ = y.shape

    a = xp.full((Nx,), 1.0 / Nx, dtype=xp.float64)
    b = xp.full((Ny,), 1.0 / Ny, dtype=xp.float64)

    # Ground cost D_ij = ||x_i - y_j||^2
    x2 = xp.sum(x * x, axis=1)[:, None]
    y2 = xp.sum(y * y, axis=1)[None, :]
    D = x2 + y2 - 2.0 * (x @ y.T)
    D = xp.maximum(D, 0.0)

    # Time geometry
    i_norm = xp.arange(Nx, dtype=xp.float64)[:, None] / Nx
    j_norm = xp.arange(Ny, dtype=xp.float64)[None, :] / Ny
    diff = i_norm - j_norm

    S = diff**2 + 1.0
    
    # Total cost: D + lambda1*S
    M_base = D + lambda1 * S
    
    # Chuáº©n hÃ³a cost giá»‘ng TAOT Ä‘á»ƒ tÄƒng tá»‘c Sinkhorn
    med = xp.median(M_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    M = M_base / med
    
    # Sinkhorn Ä‘Æ¡n giáº£n (nhanh hÆ¡n bregman_log_projection_batch)
    T = ot.sinkhorn(a, b, M, reg=lambda2, numItermax=max_iter, stopThr=tol)

    dist = xp.sum(T * M_base)
    return float(dist.get()) if on_gpu else float(dist)


## 2) t-SNE utilities + quality metrics + plotting

In [ ]:

import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE, MDS
from sklearn.neighbors import NearestNeighbors
from tslearn.datasets import UCR_UEA_datasets

# -------------------------
# t-SNE init + perplexity
# -------------------------
def suggest_perplexity(m: int) -> int:
    # sqrt(m) clipped to [5,40], also must be < (m-1)/3
    if m <= 5:
        return 2
    p = int(np.clip(np.sqrt(m), 5, 40))
    upper = max(2, (m - 1) // 3 - 1)
    return int(min(p, upper))

def tsne_from_distance(
    D: np.ndarray,
    random_state: int = 0,
    n_iter: int = 5000,
    verbose: int = 0,
):
    '''
    Two-stage: non-metric MDS init (precomputed) -> TSNE(metric='precomputed').
    Increase n_iter to improve convergence.
    '''
    m = D.shape[0]
    perplexity = suggest_perplexity(m)

    # Non-metric MDS init
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        metric=False,
        random_state=random_state,
        n_init=1,
        max_iter=300,
        normalized_stress="auto",
    )
    Y0 = mds.fit_transform(D)

    tsne = TSNE(
        n_components=2,
        metric="precomputed",
        init=Y0,
        perplexity=perplexity,
        learning_rate="auto",
        max_iter=n_iter,
        n_iter_without_progress=300,
        early_exaggeration=12.0,
        random_state=random_state,
        verbose=verbose,
        method="barnes_hut",
        angle=0.5,
    )
    Y = tsne.fit_transform(D)
    return Y, perplexity

# -------------------------
# Quality metrics from distance matrix
# -------------------------
def _knn_from_distance(D: np.ndarray, k: int):
    nn = NearestNeighbors(n_neighbors=k+1, metric="precomputed")
    nn.fit(D)
    inds = nn.kneighbors(D, return_distance=False)[:, 1:]
    return inds

def trustworthiness_from_distance(D_high: np.ndarray, Y_low: np.ndarray, k: int = 5) -> float:
    n = D_high.shape[0]
    if k >= n:
        k = n - 1

    neigh_high = _knn_from_distance(D_high, k)

    nn_low = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn_low.fit(Y_low)
    neigh_low = nn_low.kneighbors(Y_low, return_distance=False)[:, 1:]

    order = np.argsort(D_high, axis=1)
    ranks = np.empty_like(order)
    ranks[np.arange(n)[:, None], order] = np.arange(n)[None, :]

    t_sum = 0.0
    for i in range(n):
        u = [j for j in neigh_low[i] if j not in set(neigh_high[i])]
        for j in u:
            t_sum += (ranks[i, j] - k)

    denom = n * k * (2 * n - 3 * k - 1)
    return 1.0 - (2.0 / denom) * t_sum

def continuity_from_distance(D_high: np.ndarray, Y_low: np.ndarray, k: int = 5) -> float:
    n = D_high.shape[0]
    if k >= n:
        k = n - 1

    neigh_high = _knn_from_distance(D_high, k)

    nn_low = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn_low.fit(Y_low)
    neigh_low = nn_low.kneighbors(Y_low, return_distance=False)[:, 1:]

    Dy = np.sum((Y_low[:, None, :] - Y_low[None, :, :]) ** 2, axis=-1)
    order_y = np.argsort(Dy, axis=1)
    ranks_y = np.empty_like(order_y)
    ranks_y[np.arange(n)[:, None], order_y] = np.arange(n)[None, :]

    c_sum = 0.0
    for i in range(n):
        v = [j for j in neigh_high[i] if j not in set(neigh_low[i])]
        for j in v:
            c_sum += (ranks_y[i, j] - k)

    denom = n * k * (2 * n - 3 * k - 1)
    return 1.0 - (2.0 / denom) * c_sum

def lcmc_from_distance(D_high: np.ndarray, Y_low: np.ndarray, k: int = 5) -> float:
    n = D_high.shape[0]
    if k >= n:
        k = n - 1
    neigh_high = _knn_from_distance(D_high, k)

    nn_low = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn_low.fit(Y_low)
    neigh_low = nn_low.kneighbors(Y_low, return_distance=False)[:, 1:]

    shared = 0
    for i in range(n):
        shared += len(set(neigh_high[i]).intersection(set(neigh_low[i])))
    T = shared / (n * k)
    return float(T - k / (n - 1))

# -------------------------
# Plot + save
# -------------------------
def plot_tsne(Y: np.ndarray, y_true: np.ndarray, dataset_name: str, method_name: str, out_dir: str, dpi: int = 300):
    os.makedirs(out_dir, exist_ok=True)
    fig = plt.figure(figsize=(7, 6))
    
    # Convert string labels to numeric if needed
    if y_true.dtype.kind in ('U', 'S', 'O'):  # Unicode, bytes, or object
        unique_labels = np.unique(y_true)
        label_map = {label: i for i, label in enumerate(unique_labels)}
        y_numeric = np.array([label_map[label] for label in y_true])
    else:
        y_numeric = y_true
    
    plt.scatter(Y[:, 0], Y[:, 1], c=y_numeric, s=10, cmap='tab10')
    plt.title(f"{dataset_name} â€” t-SNE ({method_name})")
    plt.tight_layout()
    path = os.path.join(out_dir, f"{dataset_name}_TSNE_{method_name}.png")
    plt.savefig(path, dpi=dpi)
    plt.close(fig)
    return path

# -------------------------
# UCR loader (tslearn)
# -------------------------
def load_ucr_dataset(dataset_name: str):
    ds = UCR_UEA_datasets()
    X_train, y_train, X_test, y_test = ds.load_dataset(dataset_name)
    X = np.concatenate([X_train, X_test], axis=0)
    y = np.concatenate([y_train, y_test], axis=0)
    return X, y

def as_ragged_list(X):
    if isinstance(X, np.ndarray) and X.ndim == 3 and X.dtype != object:
        return [X[i] for i in range(X.shape[0])]
    return [np.asarray(x, dtype=float) for x in X]

def maybe_subsample(X, y, max_samples: int | None, seed: int = 0):
    if max_samples is None or len(X) <= max_samples:
        return X, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=max_samples, replace=False)
    return X[idx], y[idx]


Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7a3d19514c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error


In [ ]:
# -------------------------
# Human Action dataset loader
# -------------------------
import joblib

def load_human_action_dataset(dataset_name: str, data_dir: str = "../data/Human_Actions"):
    """
    Load Human Action datasets from pickle files.
    Supports: MSRAction3D, Weizmann, SpokenArabicDigit
    """
    import os
    from os.path import join
    
    data_path = join(data_dir, dataset_name)
    
    print(f"Loading {dataset_name} from {data_path}...")
    
    X_train = joblib.load(join(data_path, "X_train.pkl"))
    y_train = joblib.load(join(data_path, "y_train.pkl"))
    X_test = joblib.load(join(data_path, "X_test.pkl"))
    y_test = joblib.load(join(data_path, "y_test.pkl"))
    
    # Convert to numpy arrays if they are lists
    if isinstance(X_train, list):
        X_train = np.array(X_train, dtype=object)
    if isinstance(y_train, list):
        y_train = np.array(y_train)
    if isinstance(X_test, list):
        X_test = np.array(X_test, dtype=object)
    if isinstance(y_test, list):
        y_test = np.array(y_test)
    
    # Concatenate train and test
    X = np.concatenate([X_train, X_test], axis=0)
    y = np.concatenate([y_train, y_test], axis=0)
    
    print(f"Loaded {dataset_name}: {len(X)} samples, {len(np.unique(y))} classes")
    
    return X, y

def load_dataset_auto(dataset_name: str, dataset_type: str = "ucr"):
    """
    Auto-load dataset based on type.
    dataset_type: 'ucr' or 'human_action'
    """
    if dataset_type.lower() == "human_action":
        return load_human_action_dataset(dataset_name)
    else:
        return load_ucr_dataset(dataset_name)

## 3) Distance-matrix builder

In [ ]:
def _dtw_distance_series(x, y): 
    from tslearn.metrics import dtw 
    return float(dtw(x, y)) 
    
def get_distance_fn(method: str): 
    method = method.upper() 
    if method == "DTW": 
        return _dtw_distance_series 
    if method == "OPW": 
        return opw_distance_series
    if method == "TCOT":
        return tcot_distance_series 
    if method == "POW": 
        return pow_distance
    if method == "TAOT": 
        return taot_distance 
    if method == "ASW": 
        return asw_distance 
    if method == "GOW": 
        return gow_distance_series 
    if method == "OTSW": 
        return None 
        
def build_distance_matrix(
    X_list,
    method: str,
    method_kwargs: dict | None = None,
    verbose: int = 1,
):
    method_kwargs = method_kwargs or {}
    X_list = as_ragged_list(X_list)
    n = len(X_list)
    D = np.zeros((n, n), dtype=np.float64)

    if method.upper() == "OTSW":
        # --- ensemble: build 5 trees (or num_trees) and average their distance matrices ---
        num_trees = int(method_kwargs.get("num_trees", 5))
        base_seed = int(method_kwargs.get("seed", 0))

        D_sum = np.zeros((n, n), dtype=np.float64)

        for t in range(num_trees):
            model = build_otsw_tamle(
                X_list,  # Pass all sequences, not just one
                lam_time=method_kwargs.get("lam_time", 5.0),
                leaf_size=method_kwargs.get("leaf_size", 16),
                max_depth=method_kwargs.get("max_depth", 20),
                seed=base_seed + t,  # khÃ¡c seed => khÃ¡c cÃ¢y
                k_split=method_kwargs.get("k_split", 2),
                box_leaf_size=method_kwargs.get("box_leaf_size", 64),
                box_max_depth=method_kwargs.get("box_max_depth", 24),
            )

            for i in range(n):
                for j in range(i + 1, n):
                    d = otsw_between_series_fast(model, i, j)  # Pass indices, not data
                    d = float(d)
                    D_sum[i, j] += d
                    D_sum[j, i] += d

                if verbose and (i % 10 == 0 or i == n - 1):
                    print(f"[OTSW tree {t+1}/{num_trees}] done row {i+1}/{n}")

        return D_sum / num_trees

    # ---- non-OTSW branch giá»¯ nguyÃªn ----
    dist_fn = get_distance_fn(method)

    for i in range(n):
        xi = np.asarray(X_list[i], dtype=float)
        for j in range(i + 1, n):
            xj = np.asarray(X_list[j], dtype=float)
            if method.upper() == "DTW":
                d = dist_fn(xi, xj)
            else:
                d = dist_fn(xi, xj, **method_kwargs)
            D[i, j] = D[j, i] = float(d)
        if verbose and (i % 10 == 0 or i == n - 1):
            print(f"[{method.upper()}] done row {i+1}/{n}")
    return D


## 4) Main runner

In [ ]:
import os, time
import numpy as np
import pandas as pd

# Chá»‰ 2 dataset nÃ y má»›i Ä‘Æ°á»£c downsample
_DOWNSAMPLE_DATASETS = {"cincecgtorso", "mixedshapessmalltrain"}

def run_tsne_compare_all(
    datasets: list[str],
    methods: list[str],
    out_root: str = "tsne_outputs",
    n_runs: int = 5,
    k_quality: int = 5,
    tsne_n_iter: int = 5000,
    max_samples: int | None = 1000,
    random_seed: int = 0,
    method_params: dict[str, dict] | None = None,
    verbose: int = 1,
    save_every: str = "dataset",   # "dataset" | "method"
    resume: bool = True,           # náº¿u CSV Ä‘Ã£ cÃ³ thÃ¬ load lÃªn vÃ  skip cÃ¡c (dataset, method) Ä‘Ã£ cháº¡y
    dataset_type: str = "ucr",     # "ucr" or "human_action"
    downsample_spoken_arabic: bool = True,  # downsample SpokenArabicDigit to 10%
    save_embeddings: bool = True,  # LÆ°u ma tráº­n Y 2D sau khi cháº¡y t-SNE
):
    os.makedirs(out_root, exist_ok=True)
    method_params = method_params or {}

    csv_path = os.path.join(out_root, "tsne_quality_all_methods.csv")
    
    # Táº¡o thÆ° má»¥c lÆ°u embeddings
    embeddings_dir = os.path.join(out_root, "embeddings")
    if save_embeddings:
        os.makedirs(embeddings_dir, exist_ok=True)

    # ---- resume: Ä‘á»c CSV cÅ©, trÃ¡nh cháº¡y láº¡i ----
    rows: list[dict] = []
    done = set()
    if resume and os.path.exists(csv_path):
        df_old = pd.read_csv(csv_path)
        rows = df_old.to_dict("records")
        done = {(r["dataset"], r["method"]) for r in rows if "dataset" in r and "method" in r}
        if verbose:
            print(f"[RESUME] Loaded {len(rows)} rows from {csv_path}")

    def _save_csv():
        df = pd.DataFrame(rows)
        df.to_csv(csv_path, index=False)

    for dataset_name in datasets:
        if verbose:
            print("\n" + "=" * 80)
            print("DATASET:", dataset_name)

        # Load dataset based on type
        X, y = load_dataset_auto(dataset_name, dataset_type)

        # ---- Downsample logic ----
        should_downsample = False
        
        # For UCR datasets: only downsample specific datasets
        if dataset_type.lower() == "ucr":
            if (max_samples is not None) and (dataset_name.strip().lower() in _DOWNSAMPLE_DATASETS):
                should_downsample = True
                target_samples = max_samples
        
        # For Human Action datasets: downsample SpokenArabicDigit to 10%
        elif dataset_type.lower() == "human_action":
            if dataset_name == "SpokenArabicDigit" and downsample_spoken_arabic:
                should_downsample = True
                target_samples = int(len(X) * 0.1)
                if verbose:
                    print(f"[DOWNSAMPLE] {dataset_name}: {len(X)} â†’ {target_samples} samples (10%)")
        
        if should_downsample:
            X, y = maybe_subsample(X, y, max_samples=target_samples, seed=random_seed)
            if verbose and dataset_type.lower() == "ucr":
                print(f"[DOWNSAMPLE] {dataset_name} -> max_samples={target_samples}")

        X_list = as_ragged_list(X)
        n = len(X_list)
        if verbose:
            print(f"n_samples={n}, n_classes={len(np.unique(y))}")

        ds_dir = os.path.join(out_root, dataset_name)
        os.makedirs(ds_dir, exist_ok=True)

        for method in methods:
            m = method.upper()

            # skip náº¿u Ä‘Ã£ cÃ³ trong CSV (resume)
            if resume and (dataset_name, m) in done:
                if verbose:
                    print(f"\n--- METHOD: {m} (SKIP - already in CSV)")
                continue

            if verbose:
                print("\n--- METHOD:", m)

            t0 = time.time()
            D = build_distance_matrix(
                X_list,
                m,
                method_kwargs=method_params.get(m, {}),
                verbose=1 if verbose else 0,
            )
            t_dist = time.time() - t0

            trusts, conts, lcmcs = [], [], []
            t_tsne_total = 0.0
            perplexity_used = None
            Y0 = None
            
            # LÆ°u táº¥t cáº£ cÃ¡c embedding tá»« cÃ¡c runs
            all_Y = []

            for r in range(n_runs):
                seed = random_seed + r
                t1 = time.time()
                Y, perplexity = tsne_from_distance(D, random_state=seed, n_iter=tsne_n_iter, verbose=0)
                t_tsne_total += (time.time() - t1)

                if perplexity_used is None:
                    perplexity_used = perplexity
                if r == 0:
                    Y0 = Y
                
                all_Y.append(Y)

                trusts.append(trustworthiness_from_distance(D, Y, k=k_quality))
                conts.append(continuity_from_distance(D, Y, k=k_quality))
                lcmcs.append(lcmc_from_distance(D, Y, k=k_quality))

            trusts = np.array(trusts)
            conts = np.array(conts)
            lcmcs = np.array(lcmcs)

            plot_path = plot_tsne(
                Y0, y_true=y,
                dataset_name=dataset_name,
                method_name=m,
                out_dir=ds_dir,
                dpi=300,
            )
            
            # ---- LÆ°u ma tráº­n Y 2D ----
            embedding_path = None
            labels_path = None
            if save_embeddings:
                # LÆ°u embedding Ä‘áº§u tiÃªn (Y0)
                embedding_filename = f"{dataset_name}_{m}_tsne_Y.npy"
                embedding_path = os.path.join(embeddings_dir, embedding_filename)
                np.save(embedding_path, Y0)
                
                # LÆ°u táº¥t cáº£ embeddings tá»« cÃ¡c runs (n_runs, n_samples, 2)
                all_Y_filename = f"{dataset_name}_{m}_tsne_all_runs.npy"
                all_Y_path = os.path.join(embeddings_dir, all_Y_filename)
                np.save(all_Y_path, np.array(all_Y))
                
                # LÆ°u labels tÆ°Æ¡ng á»©ng
                labels_filename = f"{dataset_name}_labels.npy"
                labels_path = os.path.join(embeddings_dir, labels_filename)
                if not os.path.exists(labels_path):  # Chá»‰ lÆ°u 1 láº§n cho má»—i dataset
                    np.save(labels_path, y)
                
                if verbose:
                    print(f"Saved embedding Y0: {embedding_path}")
                    print(f"Saved all runs: {all_Y_path}")

            row = {
                "dataset": dataset_name,
                "method": m,
                "n_samples": n,
                "k_quality": k_quality,
                "n_runs": n_runs,
                "perplexity": perplexity_used,
                "tsne_n_iter": tsne_n_iter,
                "time_distance_sec": t_dist,
                "time_tsne_total_sec": t_tsne_total,
                "trust_mean": float(trusts.mean()),
                "trust_std": float(trusts.std(ddof=0)),
                "cont_mean": float(conts.mean()),
                "cont_std": float(conts.std(ddof=0)),
                "lcmc_mean": float(lcmcs.mean()),
                "lcmc_std": float(lcmcs.std(ddof=0)),
                "plot_path": plot_path,
                "embedding_path": embedding_path,
                "labels_path": labels_path,
            }
            rows.append(row)
            done.add((dataset_name, m))

            if verbose:
                print(f"Saved plot: {plot_path}")
                print(f"Trust={trusts.mean():.4f}Â±{trusts.std():.4f} | "
                      f"Cont={conts.mean():.4f}Â±{conts.std():.4f} | "
                      f"LCMC={lcmcs.mean():.4f}Â±{lcmcs.std():.4f}")

            # ---- save incremental theo method náº¿u muá»‘n ----
            if save_every == "method":
                _save_csv()
                if verbose:
                    print(f"[SAVE] CSV updated (per method): {csv_path}")

        # ---- save incremental sau má»—i dataset (Ä‘Ãºng yÃªu cáº§u báº¡n) ----
        if save_every == "dataset":
            _save_csv()
            if verbose:
                print(f"[SAVE] CSV updated (per dataset): {csv_path}")

    df = pd.DataFrame(rows)
    if verbose:
        print("\nDONE. CSV saved to:", csv_path)
    return df, csv_path

## 5) Run

In [ ]:
allDATASETS = [
    "ArrowHead",              # AH
    "BasicMotions",           # BM
    "BeetleFly",              # BF
    "CBF",                    # CBF
    "Chinatown",              # CT
    "CinCECGTorso",           # CET
    "DiatomSizeReduction",    # DSR
    "GunPointAgeSpan",        # GPA
    "GunPointMaleVersusFemale", # GPM
    "GunPointOldVersusYoung", # GPO
    "Ham",                    # Ham
    "InsectEPGRegularTrain",  # IERT
    "ItalyPowerDemand",       # IPD
    "Meat",                   # Meat
    "MelbournePedestrian",    # MP
    "MixedShapesSmallTrain",  # MS2T
    "MoteStrain",             # MS
    "OliveOil",               # O2
    "Plane",                  # Plane
    "SmoothSubspace",         # S2
]
DATASETS = [
    "ArrowHead",              # AH
    "BasicMotions",           # BM
    "BeetleFly",              # BF
    "CBF",                    # CBF
    "Chinatown",              # CT
    "CinCECGTorso",           # CET
    "DiatomSizeReduction",    # DSR
    "GunPointAgeSpan",        # GPA
    "GunPointMaleVersusFemale", # GPM
    "GunPointOldVersusYoung", # GPO
    "Ham",                    # Ham
    "InsectEPGRegularTrain",  # IERT
    "ItalyPowerDemand",       # IPD
    "Meat",                   # Meat
    "MelbournePedestrian",    # MP
    "MixedShapesSmallTrain",  # MS2T
    "MoteStrain",             # MS
    "OliveOil",               # O2
    "Plane",                  # Plane
    "SmoothSubspace",         # S2
]

allMETHODS = [
    "OPW",
    "TCOT",
    "POW",
    "TAOT",
    "ASW"
]
METHODS = [
    "OTSW"
]

METHOD_PARAMS = {
    "OPW": dict(lambda1=1.0, lambda2=0.1, sigma=0.1, use_gpu="auto", max_iter=1000, tol=5e-3),
    "TCOT": dict(lambda_pos=1.0, reg=0.1, num_iter=1000),
    "POW": dict(lam=10.0, lam_order=1.0, bandwidth=None, tolerance=5e-3),
    "TAOT": dict(lam=10.0, w=10.0, tolerance=5e-3),
    "ASW": dict(lam=10.0, auto_weight=True, w_spatial=1.0, w_order=1.0, w_struct=1.0, tolerance=5e-3),
    "GOW": dict(),
    "OTSW": dict(lam_time=5.0, leaf_size=16, max_depth=20, seed=0, k_split=2, box_leaf_size=64, box_max_depth=24),
}

# Cháº¡y t-SNE vÃ  lÆ°u cáº£ ma tráº­n Y 2D
df_results, csv_path = run_tsne_compare_all(
    datasets=DATASETS,
    methods=METHODS,
    out_root="tsne_outputs_all_methods",
    n_runs=5,
    k_quality=5,
    tsne_n_iter=5000,
    max_samples=300,
    random_seed=0,
    method_params=METHOD_PARAMS,
    verbose=1,
    save_embeddings=True,  # LÆ°u ma tráº­n Y 2D sau t-SNE
)

print(f"\n{'='*60}")
print("Káº¾T QUáº¢ ÄÃƒ LÆ¯U:")
print(f"{'='*60}")
print(f"- CSV káº¿t quáº£: {csv_path}")
print(f"- ThÆ° má»¥c embeddings: tsne_outputs_all_methods/embeddings/")
print(f"- Má»—i method cÃ³ file: {{dataset}}_{{method}}_tsne_Y.npy")
print(f"{'='*60}")

df_results.head()


DATASET: ArrowHead
n_samples=211, n_classes=3

--- METHOD: OTSW


ValueError: If ndarray, expect shape (m, n, d).

## 6) Run for Human Action Datasets

In [ ]:
# Human Action datasets
HUMAN_ACTION_DATASETS = [
    "MSRAction3D",
    "Weizmann", 
    "SpokenArabicDigit",  # Will be downsampled to 10%
]

# Methods to evaluate
HUMAN_ACTION_METHODS = [
    "DTW",
    "OPW",
    "TCOT",
    "POW",
    "TAOT",
    "ASW",
    "OTSW"
]

# Method parameters (same as UCR)
METHOD_PARAMS_HUMAN = {
    "DTW": {},
    "OPW": dict(lambda1=1.0, lambda2=0.1, sigma=0.1, use_gpu="auto", max_iter=1000, tol=5e-3),
    "TCOT": dict(lambda_pos=1.0, reg=0.1, num_iter=1000),
    "POW": dict(lam=10.0, lam_order=1.0, bandwidth=None, tolerance=5e-3),
    "TAOT": dict(lam=10.0, w=10.0, tolerance=5e-3),
    "ASW": dict(lam=10.0, auto_weight=True, w_spatial=1.0, w_order=1.0, w_struct=1.0, tolerance=5e-3),
    "OTSW": dict(lam_time=5.0, leaf_size=16, max_depth=20, seed=0, k_split=2, box_leaf_size=64, box_max_depth=24, num_trees=5),
}

# Run t-SNE evaluation on Human Action datasets
df_human_results, csv_human_path = run_tsne_compare_all(
    datasets=HUMAN_ACTION_DATASETS,
    methods=HUMAN_ACTION_METHODS,
    out_root="tsne_outputs_human_actions",
    n_runs=5,
    k_quality=5,
    tsne_n_iter=5000,
    max_samples=None,  # Not used for Human Action, handled by downsample_spoken_arabic
    random_seed=0,
    method_params=METHOD_PARAMS_HUMAN,
    verbose=1,
    dataset_type="human_action",  # Key parameter to use Human Action loader
    downsample_spoken_arabic=True,  # Downsample SpokenArabicDigit to 10%
    save_every="dataset",
    resume=True,
)

print(f"\n{'='*80}")
print("Human Action Results Summary:")
print(f"{'='*80}")
df_human_results.head(20)

## 7) Ablation Study for OTSW Parameters (t-SNE Quality)

This section performs ablation study on OTSW hyperparameters measuring t-SNE quality metrics:
- **Lambda (lam_time)**: (0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100)
- **Max Depth**: (5, 10, 15, 20, 25, 30)
- **Number of Trees**: (1, 3, 5, 7, 9, 11, 13, 15)
- **Number of Clusters (k_split)**: (2, 4, 8, 16, 32)

**Default values**: lambda=5, depth=30, trees=5, num_cluster=2

When varying one parameter, all others are fixed at default values.
Results include Trustworthiness, Continuity, LCMC, and execution time for each configuration.

In [ ]:
# ============================================================
# ABLATION STUDY for OTSW PARAMETERS (t-SNE Quality Metrics)
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tslearn.datasets import UCR_UEA_datasets

# ---------------------- Configuration ----------------------
# Default parameter values
DEFAULT_LAMBDA = 5
DEFAULT_DEPTH = 30
DEFAULT_TREES = 5
DEFAULT_NUM_CLUSTER = 2  # k_split

# Parameter ranges for ablation
LAMBDA_VALUES = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100]
DEPTH_VALUES = [5, 10, 15, 20, 25, 30]
TREES_VALUES = [1, 3, 5, 7, 9, 11, 13, 15]
NUM_CLUSTER_VALUES = [2, 4, 8, 16, 32]

# Dataset to use for ablation study
ABLATION_DATASET = "BasicMotions"  # Change this to test on different datasets
LEAF_SIZE = 16
BASE_SEED = 0
TSNE_N_ITER = 5000
K_QUALITY = 5  # k for quality metrics
ABLATION_NUM_RUNS = 5  # Number of runs per configuration

# ---------------------- Single ablation run ----------------------
def run_tsne_ablation_single(
    X_list,
    lam_time=DEFAULT_LAMBDA,
    max_depth=DEFAULT_DEPTH,
    num_trees=DEFAULT_TREES,
    k_split=DEFAULT_NUM_CLUSTER,
    seed=BASE_SEED,
    tsne_n_iter=TSNE_N_ITER,
    k_quality=K_QUALITY,
):
    """
    Run t-SNE with OTSW for a single parameter configuration.
    Returns: (Trustworthiness, Continuity, LCMC, time_total)
    """
    start_time = time.time()
    
    # Build OTSW distance matrix
    n = len(X_list)
    D_sum = np.zeros((n, n), dtype=np.float64)
    
    for t in range(num_trees):
        model = build_otsw_tamle(
            X_list,
            lam_time=lam_time,
            leaf_size=LEAF_SIZE,
            max_depth=max_depth,
            seed=seed + t,
            k_split=k_split,
        )
        
        for i in range(n):
            for j in range(i + 1, n):
                d = otsw_between_series_fast(model, i, j)
                D_sum[i, j] += d
                D_sum[j, i] += d
    
    D = D_sum / num_trees
    
    # Run t-SNE
    Y, _ = tsne_from_distance(D, random_state=seed, n_iter=tsne_n_iter)
    
    # Compute quality metrics
    trust = trustworthiness_from_distance(D, Y, k=k_quality)
    cont = continuity_from_distance(D, Y, k=k_quality)
    lcmc = lcmc_from_distance(D, Y, k=k_quality)
    
    time_total = time.time() - start_time
    
    return trust, cont, lcmc, time_total


# ---------------------- Run ablation for one parameter ----------------------
def run_tsne_ablation_for_param(X_list, param_name, param_values, num_runs=ABLATION_NUM_RUNS, **fixed_params):
    """
    Run ablation study for a single parameter with multiple runs.
    Returns: DataFrame with columns [param_value, Trust_mean, Trust_std, Cont_mean, Cont_std, LCMC_mean, LCMC_std, Time_mean, Time_std]
    """
    results = []
    
    print(f"\n{'='*60}")
    print(f"Ablation Study (t-SNE): {param_name}")
    print(f"Testing {len(param_values)} values: {param_values}")
    print(f"Number of runs per value: {num_runs}")
    print(f"Fixed params: {fixed_params}")
    print(f"{'='*60}")
    
    for val in param_values:
        params = fixed_params.copy()
        params[param_name] = val
        
        print(f"  Testing {param_name}={val}...")
        
        trust_runs = []
        cont_runs = []
        lcmc_runs = []
        time_runs = []
        
        for run_idx in range(num_runs):
            try:
                # Use different seed for each run
                params_with_seed = params.copy()
                params_with_seed['seed'] = BASE_SEED + run_idx * 100
                
                trust, cont, lcmc, time_total = run_tsne_ablation_single(X_list, **params_with_seed)
                trust_runs.append(trust)
                cont_runs.append(cont)
                lcmc_runs.append(lcmc)
                time_runs.append(time_total)
                print(f"    Run {run_idx+1}/{num_runs}: Trust={trust:.4f}, Cont={cont:.4f}, LCMC={lcmc:.4f}, Time={time_total:.2f}s")
            except Exception as e:
                print(f"    Run {run_idx+1}/{num_runs}: ERROR: {e}")
                trust_runs.append(np.nan)
                cont_runs.append(np.nan)
                lcmc_runs.append(np.nan)
                time_runs.append(np.nan)
        
        # Calculate mean and std
        trust_mean = float(np.nanmean(trust_runs))
        trust_std = float(np.nanstd(trust_runs))
        cont_mean = float(np.nanmean(cont_runs))
        cont_std = float(np.nanstd(cont_runs))
        lcmc_mean = float(np.nanmean(lcmc_runs))
        lcmc_std = float(np.nanstd(lcmc_runs))
        time_mean = float(np.nanmean(time_runs))
        time_std = float(np.nanstd(time_runs))
        
        print(f"    => Mean: Trust={trust_mean:.4f}Â±{trust_std:.4f}, Cont={cont_mean:.4f}Â±{cont_std:.4f}, LCMC={lcmc_mean:.4f}Â±{lcmc_std:.4f}, Time={time_mean:.2f}Â±{time_std:.2f}s")
        
        results.append({
            param_name: val,
            "Trust_mean": trust_mean,
            "Trust_std": trust_std,
            "Cont_mean": cont_mean,
            "Cont_std": cont_std,
            "LCMC_mean": lcmc_mean,
            "LCMC_std": lcmc_std,
            "Time_mean": time_mean,
            "Time_std": time_std,
        })
    
    return pd.DataFrame(results)


# ---------------------- Plotting function ----------------------
def plot_tsne_ablation_results(df, param_name, output_dir="ablation_results_tsne"):
    """
    Plot ablation results: Trustworthiness, Continuity, LCMC, and Time vs parameter value with error bars (std).
    """
    os.makedirs(output_dir, exist_ok=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    x = df[param_name].astype(str)
    x_numeric = range(len(x))
    
    # Plot Trustworthiness with error bars
    axes[0, 0].errorbar(x_numeric, df["Trust_mean"], yerr=df["Trust_std"], 
                        marker='o', linewidth=2, markersize=8, color='blue',
                        capsize=4, capthick=1.5, elinewidth=1.5)
    axes[0, 0].set_xlabel(param_name, fontsize=12)
    axes[0, 0].set_ylabel("Trustworthiness", fontsize=12)
    axes[0, 0].set_title(f"Trustworthiness vs {param_name}", fontsize=14)
    axes[0, 0].set_xticks(x_numeric)
    axes[0, 0].set_xticklabels(x, rotation=45, ha='right')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot Continuity with error bars
    axes[0, 1].errorbar(x_numeric, df["Cont_mean"], yerr=df["Cont_std"], 
                        marker='s', linewidth=2, markersize=8, color='green',
                        capsize=4, capthick=1.5, elinewidth=1.5)
    axes[0, 1].set_xlabel(param_name, fontsize=12)
    axes[0, 1].set_ylabel("Continuity", fontsize=12)
    axes[0, 1].set_title(f"Continuity vs {param_name}", fontsize=14)
    axes[0, 1].set_xticks(x_numeric)
    axes[0, 1].set_xticklabels(x, rotation=45, ha='right')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot LCMC with error bars
    axes[1, 0].errorbar(x_numeric, df["LCMC_mean"], yerr=df["LCMC_std"], 
                        marker='^', linewidth=2, markersize=8, color='purple',
                        capsize=4, capthick=1.5, elinewidth=1.5)
    axes[1, 0].set_xlabel(param_name, fontsize=12)
    axes[1, 0].set_ylabel("LCMC", fontsize=12)
    axes[1, 0].set_title(f"LCMC vs {param_name}", fontsize=14)
    axes[1, 0].set_xticks(x_numeric)
    axes[1, 0].set_xticklabels(x, rotation=45, ha='right')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot Time with error bars
    axes[1, 1].errorbar(x_numeric, df["Time_mean"], yerr=df["Time_std"], 
                        marker='d', linewidth=2, markersize=8, color='red',
                        capsize=4, capthick=1.5, elinewidth=1.5)
    axes[1, 1].set_xlabel(param_name, fontsize=12)
    axes[1, 1].set_ylabel("Time (seconds)", fontsize=12)
    axes[1, 1].set_title(f"Execution Time vs {param_name}", fontsize=14)
    axes[1, 1].set_xticks(x_numeric)
    axes[1, 1].set_xticklabels(x, rotation=45, ha='right')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(output_dir, f"ablation_tsne_{param_name}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"âœ… Plot saved to {fig_path}")
    return fig_path


# ---------------------- Main ablation study ----------------------
def run_full_tsne_ablation_study(dataset_name=ABLATION_DATASET, num_runs=ABLATION_NUM_RUNS):
    """
    Run complete ablation study for all OTSW Parameters measuring t-SNE quality.
    Each parameter configuration is run num_runs times (default: 5) to compute mean and std.
    """
    print(f"\n{'#'*70}")
    print(f"# OTSW ABLATION STUDY (t-SNE Quality) ON DATASET: {dataset_name}")
    print(f"{'#'*70}")
    
    # Load dataset
    ucruea = UCR_UEA_datasets()
    Xtr, ytr, Xte, yte = ucruea.load_dataset(dataset_name)
    
    # Prepare data
    X_all = np.concatenate([Xtr, Xte], axis=0)
    X_list = as_ragged_list(X_all)
    
    # Downsample to 10% if data has more than 1000 samples
    original_size = len(X_list)
    if original_size > 1000:
        sample_size = int(original_size * 0.1)
        np.random.seed(BASE_SEED)  # For reproducibility
        indices = np.random.choice(original_size, sample_size, replace=False)
        indices = np.sort(indices)  # Sort to maintain order
        
        X_list = [X_list[i] for i in indices]
        
        print(f"\nâš ï¸  Dataset downsampled: {original_size} â†’ {len(X_list)} samples (10%)")
    
    print(f"\nDataset: {dataset_name}")
    print(f"Total samples: {len(X_list)}")
    print(f"\nDefault parameters:")
    print(f"  - Lambda (lam_time): {DEFAULT_LAMBDA}")
    print(f"  - Max Depth: {DEFAULT_DEPTH}")
    print(f"  - Number of Trees: {DEFAULT_TREES}")
    print(f"  - Number of Clusters (k_split): {DEFAULT_NUM_CLUSTER}")
    print(f"  - Number of runs per config: {num_runs}")
    
    output_dir = "ablation_results_tsne"
    os.makedirs(output_dir, exist_ok=True)
    
    all_results = {}
    
    # 1. Ablation on Lambda (lam_time)
    print("\n" + "="*70)
    print("1. ABLATION ON LAMBDA (lam_time)")
    print("="*70)
    df_lambda = run_tsne_ablation_for_param(
        X_list,
        param_name="lam_time",
        param_values=LAMBDA_VALUES,
        num_runs=num_runs,
        max_depth=DEFAULT_DEPTH,
        num_trees=DEFAULT_TREES,
        k_split=DEFAULT_NUM_CLUSTER,
    )
    df_lambda.to_csv(f"{output_dir}/ablation_tsne_lambda.csv", index=False)
    plot_tsne_ablation_results(df_lambda, "lam_time", output_dir)
    all_results["lambda"] = df_lambda
    
    # 2. Ablation on Max Depth
    print("\n" + "="*70)
    print("2. ABLATION ON MAX DEPTH")
    print("="*70)
    df_depth = run_tsne_ablation_for_param(
        X_list,
        param_name="max_depth",
        param_values=DEPTH_VALUES,
        num_runs=num_runs,
        lam_time=DEFAULT_LAMBDA,
        num_trees=DEFAULT_TREES,
        k_split=DEFAULT_NUM_CLUSTER,
    )
    df_depth.to_csv(f"{output_dir}/ablation_tsne_depth.csv", index=False)
    plot_tsne_ablation_results(df_depth, "max_depth", output_dir)
    all_results["depth"] = df_depth
    
    # 3. Ablation on Number of Trees
    print("\n" + "="*70)
    print("3. ABLATION ON NUMBER OF TREES")
    print("="*70)
    df_trees = run_tsne_ablation_for_param(
        X_list,
        param_name="num_trees",
        param_values=TREES_VALUES,
        num_runs=num_runs,
        lam_time=DEFAULT_LAMBDA,
        max_depth=DEFAULT_DEPTH,
        k_split=DEFAULT_NUM_CLUSTER,
    )
    df_trees.to_csv(f"{output_dir}/ablation_tsne_trees.csv", index=False)
    plot_tsne_ablation_results(df_trees, "num_trees", output_dir)
    all_results["trees"] = df_trees
    
    # 4. Ablation on Number of Clusters (k_split)
    print("\n" + "="*70)
    print("4. ABLATION ON NUMBER OF CLUSTERS (k_split)")
    print("="*70)
    df_cluster = run_tsne_ablation_for_param(
        X_list,
        param_name="k_split",
        param_values=NUM_CLUSTER_VALUES,
        num_runs=num_runs,
        lam_time=DEFAULT_LAMBDA,
        max_depth=DEFAULT_DEPTH,
        num_trees=DEFAULT_TREES,
    )
    df_cluster.to_csv(f"{output_dir}/ablation_tsne_cluster.csv", index=False)
    plot_tsne_ablation_results(df_cluster, "k_split", output_dir)
    all_results["cluster"] = df_cluster
    
    # Summary
    print("\n" + "#"*70)
    print("# ABLATION STUDY (t-SNE Quality) COMPLETE")
    print("#"*70)
    print(f"\nResults saved to {output_dir}/ folder:")
    print("  - ablation_tsne_lambda.csv + ablation_tsne_lam_time.png")
    print("  - ablation_tsne_depth.csv + ablation_tsne_max_depth.png")
    print("  - ablation_tsne_trees.csv + ablation_tsne_num_trees.png")
    print("  - ablation_tsne_cluster.csv + ablation_tsne_k_split.png")
    
    return all_results


# ---------------------- Run the ablation study ----------------------
# Run the full ablation study (5 runs per config by default):
# all_results = run_full_tsne_ablation_study(dataset_name="BasicMotions", num_runs=5)

# Or run individual parameter ablations:
# Example: Load data first, then run only lambda ablation
# X, y = load_ucr_dataset("BasicMotions")
# X_list = as_ragged_list(X)
# df_lambda = run_tsne_ablation_for_param(X_list, "lam_time", LAMBDA_VALUES, num_runs=5, max_depth=30, num_trees=5, k_split=2)
#
# CSV output columns: param_value, Trust_mean, Trust_std, Cont_mean, Cont_std, LCMC_mean, LCMC_std, Time_mean, Time_std